# Protein Conformational Transitions calculations tutorial using BioExcel Building Blocks (biobb) and GOdMD

***
This tutorial aims to illustrate the process of computing a **conformational transition** between two known **structural conformations** of a protein, step by step, using the **BioExcel Building Blocks library (biobb)**. 

Examples shown are the calculation of the conformational transition for the **Adenylate Kinase** protein, from the **closed state** (PDB Code [1AKE](https://www.rcsb.org/structure/1AKE), [https://doi.org/10.2210/pdb1AKE/pdb](https://doi.org/10.2210/pdb1AKE/pdb)) to the **open state** (PDB Code [4AKE](https://www.rcsb.org/structure/4AKE), [https://doi.org/10.2210/pdb4AKE/pdb](https://doi.org/10.2210/pdb4AKE/pdb)). **Adenylate Kinases** are **phosphotransferase enzymes** that catalyze the interconversion of the various **adenosine phosphates** (ATP, ADP, and AMP), and are known to undergo large **conformational changes** during their **catalytic cycle**.

The code wrapped is the ***GOdMD*** method, developed in the **[Molecular Modeling and Bioinformatics](https://mmb.irbbarcelona.org/www/) group** (IRB Barcelona). **GOdMD** determines pathways for **conformational transitions** in macromolecules using **discrete molecular dynamics** and **biasing techniques** based on a combination of **essential dynamics** and **Maxwell-Demon sampling techniques**. A web implementation of the method can be found here: https://mmb.irbbarcelona.org/GOdMD/index.php

**Exploration of conformational transition pathways from coarse-grained simulations.**<br>
*Sfriso P, Hospital A, Emperador A, Orozco M.*<br>
*Bioinformatics, 129(16):1980-6.*<br>
*Available at: https://doi.org/10.1093/bioinformatics/btt324*

***

## Settings

### Biobb modules used

 - [biobb_godmd](https://github.com/bioexcel/biobb_godmd): Tools to compute protein conformational transitions using GOdMD.
 - [biobb_io](https://github.com/bioexcel/biobb_io): Tools to fetch biomolecular data from public databases.
 - [biobb_structure_utils](https://github.com/bioexcel/biobb_structure_utils): Tools to modify or extract information from a PDB structure.
 - [biobb_analysis](https://github.com/bioexcel/biobb_analysis): Tools to analyse Molecular Dynamics trajectories.
  
### Auxiliary libraries used

* [emboss](https://www.ebi.ac.uk/Tools/emboss/): Software that automatically copes with data in a variety of formats and even allows transparent retrieval of sequence data from the web.
* [jupyter](https://jupyter.org/): Free software, open standards, and web services for interactive computing across all programming languages.
* [molviewspec](https://molstar.org/mol-view-spec/): Python toolkit to describe molecular scenes and display them with the [Mol*](https://molstar.org/) viewer in notebooks.

### Conda Installation and Launch

```console
git clone https://github.com/bioexcel/biobb_wf_godmd.git
cd biobb_wf_godmd
conda env create -f conda_env/environment.yml
conda activate biobb_wf_godmd
jupyter-notebook biobb_wf_godmd/notebooks/biobb_wf_godmd.ipynb
``` 

***
## Pipeline steps
 1. [Input Parameters](#input)
 2. [Fetching Structures](#fetch)
 3. [Preparing Structures](#preparing)
 3. [Residue Mapping](#mapping)
 4. [Conformational Transition](#godmd)
 5. [Transition Visualization](#trajectory)
 6. [Questions & Comments](#questions)
 
***

<img src="https://bioexcel.eu/wp-content/uploads/2019/04/Bioexcell_logo_1080px_transp.png" alt="Bioexcel2 logo" width="400" >

***

## Initializing colab
The cell below is used only in case this notebook is executed via **Google Colab**. This process can take a **few minutes** since **miniforge** is **downloaded** and **installed** first, and then the **conda environment** must be created.

In [ ]:
# Only executed when using google colab
import sys
import os
if 'google.colab' in sys.modules:
  !git clone -b molstar-viewer https://github.com/bioexcel/biobb_wf_godmd.git
  !wget https://github.com/conda-forge/miniforge/releases/download/26.1.0-0/Miniforge3-26.1.0-0-Linux-x86_64.sh
  !bash Miniforge3-26.1.0-0-Linux-x86_64.sh -b -p /usr/local/miniforge
  !/usr/local/miniforge/condabin/mamba create -f biobb_wf_godmd/conda_env/environment.yml -y
  %env PATH=/usr/local/bin:/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/miniforge/envs/biobb_wf_godmd/bin
  _ = (sys.path.append("/usr/local/miniforge/envs/biobb_wf_godmd/lib/python3.12/site-packages/"))
  os.chdir('/content/biobb_wf_godmd/biobb_wf_godmd/notebooks')

<a id="input"></a>
## Input parameters
**Input parameters** needed:
 - **y libraries**: Libraries to be used in the workflow are imported once at the beginning
 
 
 - **pdbOrigin**: PDB code for the origin structure (e.g. 1AKE, [https://doi.org/10.2210/pdb1AKE/pdb](https://doi.org/10.2210/pdb1AKE/pdb))
 - **chainOrigin**: Chain for the origin structure (e.g. A)
 - **ligandOrigin**: Ligand (if present) for the origin structure (e.g. AP5, Drugbank [DB01717](https://go.drugbank.com/drugs/DB01717)) 
 
 
 - **pdbTarget**: PDB code for the target structure (e.g. 4AKE, [https://doi.org/10.2210/pdb4AKE/pdb](https://doi.org/10.2210/pdb4AKE/pdb))
 - **chainTarget**: Chain for the target structure (e.g. A)
 - **ligandTarget**: Ligand (if present) for the target structure (e.g. None)


In [ ]:
import os
import json
import subprocess
import uuid
from pathlib import Path
from IPython.display import HTML, Javascript, display
import molviewspec as mvs

def show_side_by_side(views, height=500):
    """Show Mol* viewers in a row. views: list of (builder, data) pairs, as passed to molstar_notebook"""
    ids = [f'molstar_{uuid.uuid4().hex}' for _ in views]
    display(HTML('<div style="display:flex;gap:4px">' + ''.join(f'<div id="{i}" style="flex:1;min-width:0"></div>' for i in ids) + '</div>'))
    js = ''
    for div_id, (builder, data) in zip(ids, views):
        page = json.dumps(mvs.molstar_html(builder, data=data)).replace('</script>', '</" + "script" + ">')
        js += (f'var frame = document.createElement("iframe"); frame.style = "border:0;width:100%;height:{height}px";'
               f'frame.src = URL.createObjectURL(new Blob([{page}], {{type: "text/html"}}));'
               f'document.getElementById("{div_id}").appendChild(frame);')
    display(Javascript('setTimeout(function(){' + js + '}, 100);'))

# Adenylate Kinase (ADK)
pdbOrigin = "1ake" 
chainOrigin = "A"
ligandOrigin = "AP5"

pdbTarget = "4ake" 
chainTarget = "A"
ligandTarget = None

# Other Examples (taken from https://mmb.irbbarcelona.org/TransAtlas/)

# Estrogen Receptor Alpha (ERα)
# pdbOrigin = "1a52" 
# pdbTarget = "3dt3" 
# chainOrigin = "A"
# chainTarget = "B"
# ligandOrigin = "EST"
# ligandTarget = "369"

# Calmodulin (CaM)
# pdbOrigin = "1deg" 
# pdbTarget = "2f2o" 
# chainOrigin = "A"
# chainTarget = "A"
# ligandOrigin = None
# ligandTarget = None

<a id="fetch"></a>
***
## Fetching PDB structures
Downloading **PDB structures** with the origin and target **protein molecules** from the RCSB PDB database.<br>
Alternatively, **PDB files** can be used as starting structures. <br>
***
**Building Blocks** used:
 - [Pdb](https://biobb-io.readthedocs.io/en/latest/api.html#module-api.pdb) from **biobb_io.api.pdb**
***

In [ ]:
# Import module
from biobb_io.api.pdb import pdb

# Create properties dict and inputs/outputs
originPDB = pdbOrigin+'.pdb'
targetPDB = pdbTarget+'.pdb'

prop_origin = {
    'pdb_code': pdbOrigin,
    'filter': False
}

prop_target = {
    'pdb_code': pdbTarget,
    'filter': False
}

# Launch bb for Origin PDB
pdb(output_pdb_path=originPDB,
    properties=prop_origin)

# Launch bb for Target PDB
pdb(output_pdb_path=targetPDB,
    properties=prop_target)

<a id="vis3D"></a>
### Visualizing 3D structures
Visualizing the downloaded/given **PDB structures** using **Mol\***: 

In [ ]:
# Show different structures (for comparison)

views = []
for pdb_file in [originPDB, targetPDB]:
    builder = mvs.create_builder()
    structure = builder.download(url=pdb_file).parse(format='pdb').model_structure()
    structure.component(selector='polymer').representation(type='cartoon').color(custom={'molstar_color_theme_name': 'secondary-structure'})
    structure.component(selector='ligand').representation(type='ball_and_stick').color(custom={'molstar_color_theme_name': 'element-symbol'})
    views.append((builder, {pdb_file: Path(pdb_file).read_bytes()}))
show_side_by_side(views, height=400)

<a id="preparing"></a>
***
## Preparing the structures
Preparing the **structures** to be used for the **conformational transition** calculation. 
- Extracting the interesting **chains** from the **PDB structures** (chain A in both cases).
- Removing the **inhibitor** AP5A from the **closed conformation**. 
***
**Building Blocks** used:
 - [extract_chain](https://biobb-structure-utils.readthedocs.io/en/stable/utils.html#module-utils.extract_chain) from **biobb_structure_utils.utils.extract_chain**
 - [remove_molecules](https://biobb-structure-utils.readthedocs.io/en/stable/utils.html#module-utils.remove_molecules) from **biobb_structure_utils.utils.remove_molecules**
 
***

Extracting the interesting **chains** from the **PDB structures** (chain A in both cases).

In [ ]:
from biobb_structure_utils.utils.extract_chain import extract_chain

originPDB_chain = pdbOrigin + ".chains.pdb"
targetPDB_chain = pdbTarget + ".chains.pdb"

prop = {
    'chains': [ chainOrigin ]
}


# Launch bb for Origin PDB
extract_chain(
    input_structure_path=originPDB,
    output_structure_path=originPDB_chain,
    properties=prop
)

prop = {
    'chains': [ chainTarget ]
}

# Launch bb for Target PDB
extract_chain(
    input_structure_path=targetPDB,
    output_structure_path=targetPDB_chain,
    properties=prop
)

Removing the **inhibitor** AP5A from the **closed conformation**. 

In [ ]:
from biobb_structure_utils.utils.remove_molecules import remove_molecules

originPDB_chain_nolig = pdbOrigin + ".chains.nolig.pdb"
targetPDB_chain_nolig = pdbTarget + ".chains.nolig.pdb"

if ligandOrigin:
    prop = {
        'molecules': [
            {
                'name' : ligandOrigin
            }
        ]
    }
    remove_molecules(input_structure_path=originPDB_chain,
                     output_molecules_path=originPDB_chain_nolig,
                     properties=prop)
else:
    originPDB_chain_nolig = originPDB_chain

if ligandTarget:
    prop = {
        'molecules': [
            {
                'name' : ligandTarget
            }
        ]
    }
    remove_molecules(input_structure_path=targetPDB_chain,
                     output_molecules_path=targetPDB_chain_nolig,
                     properties=prop)
else:
    targetPDB_chain_nolig = targetPDB_chain


<a id="vis3D_chains"></a>
### Visualizing 3D structures
Visualizing the modified **PDB structures** using **Mol\***: 

In [ ]:
# Show different structures generated (for comparison)

views = []
for pdb_file in [originPDB_chain_nolig, targetPDB_chain_nolig]:
    builder = mvs.create_builder()
    structure = builder.download(url=pdb_file).parse(format='pdb').model_structure()
    structure.component(selector='polymer').representation(type='cartoon').color(custom={'molstar_color_theme_name': 'secondary-structure'})
    structure.component(selector='ligand').representation(type='ball_and_stick').color(custom={'molstar_color_theme_name': 'element-symbol'})
    views.append((builder, {pdb_file: Path(pdb_file).read_bytes()}))
show_side_by_side(views, height=400)

<a id="mapping"></a>
***
## Computing the mapping
**GOdMD** works by moving the atoms from the position of the **origin structure** to the one in the **target structure**. For that, a one-to-one correspondance is needed. This step builds a couple of **mapping files** (.aln) from an internal **sequence alignment** (using ***[EMBOSS water](https://www.ebi.ac.uk/Tools/psa/emboss_water/)*** pairwise sequence alignment tool).     
***
**Building Blocks** used:
 - [godmd_prep](https://biobb-godmd.readthedocs.io/en/latest/godmd.html#module-godmd.godmd_prep) from **biobb_godmd.godmd.godmd_prep**
 
***

In [ ]:
from biobb_godmd.godmd.godmd_prep import godmd_prep

originALN = pdbOrigin + ".aln"
targetALN = pdbTarget + ".aln"

prop = {
    'gapopen' : '12.0',
    'gapextend' : '2'
}

godmd_prep( input_pdb_orig_path=originPDB_chain_nolig,
            input_pdb_target_path=targetPDB_chain_nolig,
            output_aln_orig_path=originALN,
            output_aln_target_path=targetALN,
            properties=prop)

<a id="godmd"></a>
***
## Running GOdMD 
Computing the **conformational transition**, from the **origin** to the **target** structure, using **GOdMD** and the **mappings** generated in the previous step. The output file is a **trajectory file** in **mdcrd** format.
    
***
**Building Blocks** used:
 - [godmd_run](https://biobb-godmd.readthedocs.io/en/latest/godmd.html#module-godmd.godmd_run) from **biobb_godmd.godmd.godmd_run**
 
***

In [ ]:
from biobb_godmd.godmd.godmd_run import godmd_run

godmd_log = pdbOrigin + "-" + pdbTarget + ".godmd.log"
godmd_trj = pdbOrigin + "-" + pdbTarget + ".godmd.mdcrd"
godmd_ene = pdbOrigin + "-" + pdbTarget + ".godmd.ene.out"
godmd_pdb = pdbOrigin + "-" + pdbTarget + ".godmd.pdb"

prop = {
    'godmdin':{
        'temp' : 400
    }
}

godmd_run(   input_pdb_orig_path=originPDB_chain_nolig,
             input_pdb_target_path=targetPDB_chain_nolig,
             input_aln_orig_path=originALN,
             input_aln_target_path=targetALN,
             output_log_path=godmd_log,
             output_ene_path=godmd_ene,
             output_trj_path=godmd_trj,
             output_pdb_path=godmd_pdb,
             properties=prop)


<a id="xtc"></a>
***
## Converting trajectory to XTC (visualization)
Converting the generated **GOdMD trajectory** from the **mdcrd** format to a **xtc** format, for the sake of visualization with **Mol\*** (see next cell).<br> <br>
***
**Building Blocks** used:
 - [cpptraj_convert](https://biobb-analysis.readthedocs.io/en/latest/ambertools.html#module-ambertools.cpptraj_convert) from **biobb_analysis.ambertools.cpptraj_convert**
***

In [ ]:
from biobb_analysis.ambertools.cpptraj_convert import cpptraj_convert

godmd_trj_xtc = pdbOrigin + "-" + pdbTarget + ".godmd.xtc"

prop = {
    'format': 'xtc'
}

cpptraj_convert(input_top_path=godmd_pdb,
                input_traj_path=godmd_trj,
                output_cpptraj_path=godmd_trj_xtc,
                properties=prop)


***
<a id="trajectory"></a>
## Visualizing trajectory
Visualizing the **GOdMD** computed **conformational transition** using **Mol\***. The scene shows the **origin** (tube, blue) and **target** (tube, red) structure for reference. 

***

In [ ]:
# Show trajectory

# Number of frames of the GOdMD trajectory (from GROMACS gmx check)
check = subprocess.run(['gmx', 'check', '-f', godmd_trj_xtc], capture_output=True, text=True)
n_frames = int(next(line.split()[1] for line in check.stderr.splitlines() if line.startswith('Coords')))

builder = mvs.create_builder()
# GOdMD trajectory coordinates
builder.download(url=godmd_trj_xtc).parse(format='xtc').coordinates(ref='trajectory')
# GOdMD structure used as topology (origin CA atoms, already fitted onto the target)
godmd = builder.download(url=godmd_pdb).parse(format='pdb')
structure = godmd.model_structure(coordinates_ref='trajectory', ref='structure')
structure.component(selector='all').representation(type='cartoon').color(custom={'molstar_color_theme_name': 'sequence-id'})

# Origin Structure
origin = godmd.model_structure()
origin.component(selector='all').representation(type='cartoon').color(color='blue').opacity(opacity=0.2)

# Target Structure
target = builder.download(url=targetPDB_chain_nolig).parse(format='pdb').model_structure()
target.component(selector='polymer').representation(type='cartoon').color(color='red').opacity(opacity=0.2)

# Play the trajectory in a loop (100 ms per frame)
builder.animation(autoplay=True, loop=True).interpolate(
    kind='scalar', target_ref='structure', property='model_index',
    start=0, end=n_frames - 1, discrete=True, duration_ms=n_frames * 100
)
data = {
    godmd_trj_xtc: Path(godmd_trj_xtc).read_bytes(),
    godmd_pdb: Path(godmd_pdb).read_bytes(),
    targetPDB_chain_nolig: Path(targetPDB_chain_nolig).read_bytes()
}
builder.molstar_notebook(data=data, width='100%', height=600)

***
<a id="questions"></a>

## Questions & Comments

Questions, issues, suggestions and comments are really welcome!

* GitHub issues:
    * [https://github.com/bioexcel/biobb](https://github.com/bioexcel/biobb)

* BioExcel forum:
    * [https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library](https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library)
